# Helical Spring ML: Stress and Deformation Prediction

This notebook trains regression models using the 100 spring designs with complete ANSYS results. Each spring has results at 500 N, 1000 N, and 1500 N, so the prepared dataset contains 300 spring-load rows.

**Important:** The test split is grouped by spring ID. All load cases for a spring stay together to avoid leakage. Predictions are estimates of ANSYS results and should be validated before engineering use.

In [1]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import joblib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Joblib:", joblib.__version__)

print("SUCCESS: All required libraries are working!")

Pandas: 3.0.6
NumPy: 2.5.4
Scikit-learn: 1.9.1
Matplotlib: 3.11.1
Joblib: 1.6.0
SUCCESS: All required libraries are working!


## 1. Upload the prepared dataset

Upload `spring_ml_dataset_100_springs.csv` when prompted. This file contains only the 100 complete spring designs and their three load cases.

In [1]:
import os

DATA_FILE = "spring_ml_dataset_100_springs.csv"

print("Working directory:", os.getcwd())
print("Dataset found:", os.path.exists(DATA_FILE))

Working directory: C:\Users\advai\Documents\FALL 2026-27\MACHINE ELEMENTS\Spring_ML_Project
Dataset found: True


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_FILE)
print("Rows:", len(df))
print("Unique spring designs:", df["Spring_ID"].nunique())
display(df.head())
print("\nRows per material:")
display(df["Material"].value_counts())


Rows: 300
Unique spring designs: 100


,Spring_ID,Free_Length_mm,Outer_Diameter_mm,Inner_Diameter_mm,Mean_Coil_Diameter_mm,Pitch_mm,Active_Coils,Wire_Diameter_mm,Material,Applied_Force_N,Total_Deformation_mm,Von_Mises_Stress_MPa
0,Spring_001,85.0,46.0,32.0,39.0,15.0,5.0,6.0,Mild Steel,500,9.9628,913.62
1,Spring_001,85.0,46.0,32.0,39.0,15.0,5.0,6.0,Mild Steel,1000,19.9260,1827.20
2,Spring_001,85.0,46.0,32.0,39.0,15.0,5.0,6.0,Mild Steel,1500,29.8880,2740.80
3,Spring_002,118.0,51.0,37.0,44.0,23.0,5.0,7.0,Mild Steel,500,7.6948,732.83
4,Spring_002,118.0,51.0,37.0,44.0,23.0,5.0,7.0,Mild Steel,1000,15.3900,1465.70



Rows per material:


Material
Mild Steel             138
Stainless Steel 304     60
Structural Steel        36
Stainless Steel 316     24
Titanium Alloy          24
Aluminum Alloy          18
Name: count, dtype: int64

## 2. Train and compare models

Inputs are spring geometry, material, and applied force. Outputs are total deformation (mm) and von Mises stress (MPa). We compare Random Forest and Extra Trees using a held-out set of spring designs.

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numeric_features = [
    "Free_Length_mm", "Outer_Diameter_mm", "Inner_Diameter_mm",
    "Mean_Coil_Diameter_mm", "Pitch_mm", "Active_Coils",
    "Wire_Diameter_mm", "Applied_Force_N"
]
categorical_features = ["Material"]
target_cols = ["Total_Deformation_mm", "Von_Mises_Stress_MPa"]
group_col = "Spring_ID"

df = df.dropna(subset=numeric_features + categorical_features + target_cols + [group_col]).copy()
X = df[numeric_features + categorical_features]
y = df[target_cols]
groups = df[group_col]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Training springs:", groups.iloc[train_idx].nunique())
print("Testing springs:", groups.iloc[test_idx].nunique())


Training springs: 80
Testing springs: 20


In [8]:
preprocess = ColumnTransformer(
    transformers=[
        ("material", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=500, random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=500, random_state=42, n_jobs=-1
    )
}

trained = {}
rows = []

for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    trained[name] = pipe

    for i, target in enumerate(target_cols):
        rows.append({
            "Model": name,
            "Target": target,
            "MAE": mean_absolute_error(y_test.iloc[:, i], pred[:, i]),
            "RMSE": np.sqrt(mean_squared_error(y_test.iloc[:, i], pred[:, i])),
            "R2": r2_score(y_test.iloc[:, i], pred[:, i])
        })

metrics = pd.DataFrame(rows).sort_values(["Target", "RMSE"])
display(metrics)


,Model,Target,MAE,RMSE,R2
0,Random Forest,Total_Deformation_mm,14.972555,20.043050,0.521512
2,Extra Trees,Total_Deformation_mm,15.915726,20.781202,0.485619
1,Random Forest,Von_Mises_Stress_MPa,569.115653,787.482656,0.612724
3,Extra Trees,Von_Mises_Stress_MPa,712.536318,957.072201,0.427959


## 3. Train final models on all 100 springs

After evaluating the held-out test set, train final models on all available spring designs for later prediction. The test metrics above remain the honest estimate of performance for the chosen split.

In [9]:
final_models = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
    pipe.fit(X, y)
    final_models[name] = pipe

import joblib
for name, pipe in final_models.items():
    filename = name.lower().replace(" ", "_") + "_spring_model.joblib"
    joblib.dump(pipe, filename)

metrics.to_csv("spring_ml_test_metrics.csv", index=False)
print("Saved final model files and spring_ml_test_metrics.csv.")


Saved final model files and spring_ml_test_metrics.csv.


## 4. Predict a new spring

Edit the example values below. Use the same units as the dataset. The material name must match a known category where possible. Predictions are most defensible for designs and forces within the training data's range.

In [30]:
new_spring = pd.DataFrame([{
    "Free_Length_mm": 157.9,
    "Outer_Diameter_mm": 40.2,
    "Inner_Diameter_mm": 36.2,
    "Mean_Coil_Diameter_mm": 43.2,
    "Pitch_mm": 12,
    "Active_Coils": 5.0,
    "Wire_Diameter_mm": 10.0,
    "Material": "Aluminum Alloy",
    "Applied_Force_N": 1000
}])

# Change the model name if the test metrics indicate another model is better.
chosen_model = final_models["Random Forest"]
prediction = chosen_model.predict(new_spring)[0]

print(f"Predicted total deformation: {prediction[0]:.4f} mm")
print(f"Predicted von Mises stress: {prediction[1]:.2f} MPa")


Predicted total deformation: 15.0085 mm
Predicted von Mises stress: 1562.72 MPa
